# API

> JSON API for the Observable frontend (`frontend/`).


In [ ]:
#| default_exp api


In [ ]:
#| export
from pathlib import Path
from fasthtml.common import *
from starlette.middleware.cors import CORSMiddleware
from starlette.routing import Mount, Route
from starlette.staticfiles import StaticFiles
from iverson.core import asset_dir, checkout_root


In [ ]:
#| export
def mount(app, rt, ns):
    "Add the routes. `ns` is main.py's globals()."
    sounds, SP, G, SECS = (ns[k] for k in 'sounds SP G SECS'.split())
    Sound, skey, snd, set_number, typable, typed, lab, cps = (
        ns[k] for k in 'Sound skey snd set_number typable typed lab cps'.split())
    talk_key, set_talk_key = ns['talk_key'], ns['set_talk_key']

    app.add_middleware(CORSMiddleware, allow_origin_regex=r'http://(localhost|127\.0\.0\.1):\d+', allow_methods=['*'], allow_headers=['*'])
    dist = (checkout_root() or asset_dir())/'frontend'/'dist'
    if dist.exists():
        async def home(request): return FileResponse(dist/'index.html')
        async def go_root(request): return RedirectResponse('/', status_code=303)
        for name in ('_observablehq', '_import', '_npm', '_file'):
            sub = dist/name
            if sub.exists(): app.router.routes.insert(0, Mount(f'/{name}', app=StaticFiles(directory=sub), name=f'fe-{name}'))
        app.router.routes.insert(0, Route('/', endpoint=home))
        app.router.routes.insert(0, Route('/app', endpoint=go_root))
        app.router.routes.insert(0, Route('/app/', endpoint=go_root))

    def sj(s):
        k = skey(s)
        return dict(id=s.id, name=s.name, glyph=s.glyph, key=s.key, types=k, label=lab(k) if k else '', cps=cps(k) if k else '', n=0,
                    gk=typed(s.glyph) if s.glyph in G else '')

    def talk_options():
        from iverson.keys import TALK_KEY_LABELS
        return [dict(id=k, label=lab) for k,lab in TALK_KEY_LABELS]

    @rt('/api/state')
    def api_state():
        return dict(sounds=[sj(s) for s in sounds()], sp=SP, talk_key=talk_key(), talk_keys=talk_options())

    @rt('/api/glyphs')
    def api_glyphs(): return JSONResponse([dict(title=t, glyphs=[dict(sym=g.sym, mean=g.mean, dev=g.dev, iast=g.iast, ename=getattr(g, 'ename', '') or '') for g in gs]) for t,gs in SECS])

    @rt('/api/assign')
    def api_assign(sym:str, name:str=''):
        msg, err = set_number(sym, name)
        return dict(ok=not err, msg=msg)

    @rt('/api/setkey')
    def api_setkey(id:int, f:str, v:str=''):
        s, v = sounds[id], v.strip()
        if f == 's': k = '' if v == '-' else v
        elif not v: k = s.key if s.key in SP else ''
        else: k = v if typable(v) and v not in SP else s.key
        sounds.update(id=id, key=k)
        return dict(ok=True)

    @rt('/api/talkkey')
    def api_talkkey(v:str='ctrl'):
        "Set the hold-to-talk key used by `iverson-receive`."
        cur = set_talk_key(v)
        return dict(ok=True, talk_key=cur, talk_keys=talk_options())

    @rt('/api/rename')
    def api_rename(id:int, v:str):
        norm = ns['normalize_code'](v)
        if norm is None: return dict(ok=False, msg='Enter 1–3 digits (stored as a 3-digit code, e.g. 001).')
        other = first(s for s in sounds() if s.name == norm and s.id != id)
        if other: return dict(ok=False, msg=f'Number {norm} is already used by {other.glyph or other.name}.')
        sounds.update(id=id, name=norm); return dict(ok=True, name=norm)

    @rt('/api/forget')
    def api_forget(id:int): sounds.delete(id); return dict(ok=True)

    @rt('/api/forgetall')
    def api_forgetall(): [sounds.delete(s.id) for s in sounds()]; return dict(ok=True)



In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
